# 3. Encoding Categorical Variables: Titanic & LendingClub

Categorical variables are columns with text-based categories that must be converted to numbers before feeding into ML models.

Common encoding approaches:

1. **One-hot encoding** — creates a binary column per category  
2. **Label encoding** — assigns an integer to each category  
3. **Ordinal encoding** — assigns integers respecting a defined order  
4. **Target encoding** — replaces a category with the mean of the target variable  
5. **Frequency encoding** — replaces a category with how often it appears  

For Titanic (OpenML), useful categorical columns include `sex`, `embarked`, `pclass`, and `cabin`.  
For LendingClub (OpenML), the only categorical column in this version is `purpose`.

In [1]:
# ============================================================
# Section 3: Encoding Categorical Variables - Titanic
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split



In [2]:

# 1. Load Titanic Dataset from OpenML
print('Fetching Titanic dataset from OpenML...')
titanic = fetch_openml(name='titanic', version=1, as_frame=True, parser='auto')
titanic_df = titanic.frame

# 2. Load LendingClub Dataset via OpenML
print('Fetching LendingClub dataset from OpenML...')
# Using a common OpenML version of lending club
lending = fetch_openml(name='lending-club-loan-data', version=1, as_frame=True, parser='auto')
lending_df = lending.frame

print('Datasets loaded successfully!')

Fetching Titanic dataset from OpenML...
Fetching LendingClub dataset from OpenML...
Datasets loaded successfully!


In [3]:
# Convert pclass to categorical because it represents passenger class, not a continuous number
if "pclass" in titanic_df.columns:
    titanic_df["pclass"] = titanic_df["pclass"].astype("category")

categorical_cols = titanic_df.select_dtypes(
    include=["object", "str", "category", "bool"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

Categorical columns:
['pclass', 'survived', 'name', 'sex', 'ticket', 'cabin', 'embarked', 'boat', 'home.dest']


In [4]:
# Identify categorical columns

for col in categorical_cols:
    print("\n", col)
    display(titanic_df[col].value_counts(dropna=False).head(5))


 pclass


pclass
3    709
1    323
2    277
Name: count, dtype: int64


 survived


survived
0    809
1    500
Name: count, dtype: int64


 name


...

embarked
S      914
C      270
Q      123
NaN      2
Name: count, dtype: int64


 boat


boat
NaN    823
13      39
C       38
15      37
14      33
Name: count, dtype: int64


 home.dest


home.dest
NaN              564
New York, NY      64
London            14
Montreal, PQ      10
Paris, France      9
Name: count, dtype: int64

### Clean Categorical Columns

Before encoding:

- Fill missing values
- Convert to lowercase
- Strip spaces
- Standardize values

In [5]:
# Clean categorical columns

for col in titanic_df.select_dtypes(include=["object", "str", "category"]).columns:
    titanic_df[col] = titanic_df[col].astype("object")
    titanic_df[col] = titanic_df[col].fillna("missing")
    titanic_df[col] = titanic_df[col].astype(str).str.strip().str.lower()

for col in titanic_df.select_dtypes(include=["bool"]).columns:
    titanic_df[col] = titanic_df[col].fillna(False)

display(titanic_df[categorical_cols].head())

,pclass,survived,name,sex,ticket,cabin,embarked,boat,home.dest
0,1,1,"allen, miss. elisabeth walton",female,24160,b5,s,2,"st louis, mo"
1,1,1,"allison, master. hudson trevor",male,113781,c22 c26,s,11,"montreal, pq / chesterville, on"
2,1,0,"allison, miss. helen loraine",female,113781,c22 c26,s,missing,"montreal, pq / chesterville, on"
3,1,0,"allison, mr. hudson joshua creighton",male,113781,c22 c26,s,missing,"montreal, pq / chesterville, on"
4,1,0,"allison, mrs. hudson j c (bessie waldo daniels)",female,113781,c22 c26,s,missing,"montreal, pq / chesterville, on"


### One-Hot Encoding

One-hot encoding creates one binary column per unique category.

Use it for **low-cardinality** nominal columns where categories have no meaningful order.

Good candidates in Titanic:

- `sex` (female/male)
- `embarked` (C/Q/S)
- `pclass` (1/2/3)

In [6]:
# Select low-cardinality columns for one-hot encoding

onehot_cols = [
    col for col in categorical_cols
    if titanic_df[col].nunique() <= 10
]

print("Columns selected for one-hot encoding:")
print(onehot_cols)

Columns selected for one-hot encoding:
['pclass', 'survived', 'sex', 'embarked']


In [7]:
# One-hot encoding using pandas

titanic_onehot = pd.get_dummies(
    titanic_df,
    columns=onehot_cols,
    drop_first=True,
    dtype=int
)

print("Original shape:", titanic_df.shape)
print("After one-hot encoding:", titanic_onehot.shape)

display(titanic_onehot.head())

Original shape: (1309, 14)
After one-hot encoding: (1309, 17)


,name,age,sibsp,parch,ticket,fare,cabin,boat,body,home.dest,pclass_2,pclass_3,survived_1,sex_male,embarked_missing,embarked_q,embarked_s
0,"allen, miss. elisabeth walton",29.0000,0,0,24160,211.3375,b5,2,NaN,"st louis, mo",0,0,1,0,0,0,1
1,"allison, master. hudson trevor",0.9167,1,2,113781,151.5500,c22 c26,11,NaN,"montreal, pq / chesterville, on",0,0,1,1,0,0,1
2,"allison, miss. helen loraine",2.0000,1,2,113781,151.5500,c22 c26,missing,NaN,"montreal, pq / chesterville, on",0,0,0,0,0,0,1
3,"allison, mr. hudson joshua creighton",30.0000,1,2,113781,151.5500,c22 c26,missing,135.0,"montreal, pq / chesterville, on",0,0,0,1,0,0,1
4,"allison, mrs. hudson j c (bessie waldo daniels)",25.0000,1,2,113781,151.5500,c22 c26,missing,NaN,"montreal, pq / chesterville, on",0,0,0,0,0,0,1


In [8]:
# One-hot encoding using sklearn

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    drop="first"
)

encoded_array = encoder.fit_transform(titanic_df[onehot_cols])

encoded_col_names = encoder.get_feature_names_out(onehot_cols)

titanic_encoded_sklearn = pd.DataFrame(
    encoded_array,
    columns=encoded_col_names,
    index=titanic_df.index
)

display(titanic_encoded_sklearn.head())

,pclass_2,pclass_3,survived_1,sex_male,embarked_missing,embarked_q,embarked_s
0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,0.0,0.0,1.0,1.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0


### Label Encoding

Label encoding assigns each category an integer.

This can be useful for tree-based models, but should be used carefully for linear models because it may create a false ordering.

In [9]:
# Label encode selected columns

label_cols = ["sex", "embarked"]

label_cols = [col for col in label_cols if col in titanic_df.columns]

titanic_label = titanic_df.copy()

label_encoders = {}

for col in label_cols:
    le = LabelEncoder()
    titanic_label[col + "_label"] = le.fit_transform(titanic_label[col].astype(str))
    label_encoders[col] = le
    
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"{col} mapping:", mapping)

display(titanic_label[[*label_cols, *[col + "_label" for col in label_cols]]].head())

sex mapping: {'female': np.int64(0), 'male': np.int64(1)}
embarked mapping: {'c': np.int64(0), 'missing': np.int64(1), 'q': np.int64(2), 's': np.int64(3)}


,sex,embarked,sex_label,embarked_label
0,female,s,0,3
1,male,s,1,3
2,female,s,0,3
3,male,s,1,3
4,female,s,0,3


### Ordinal Encoding

Ordinal encoding is used when categories have a natural order.

For Titanic, passenger class has a natural order:

First class > Second class > Third class

In [10]:
# Ordinal encoding for passenger class if available

titanic_ordinal = titanic_df.copy()

if "pclass" in titanic_ordinal.columns:
    # pclass is already numeric in many Titanic datasets
    titanic_ordinal["pclass_ordinal"] = titanic_ordinal["pclass"]

if "class" in titanic_ordinal.columns:
    class_order = [["third", "second", "first", "missing"]]
    
    ordinal_encoder = OrdinalEncoder(
        categories=class_order,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )
    
    titanic_ordinal["class_ordinal"] = ordinal_encoder.fit_transform(
        titanic_ordinal[["class"]]
    )

display(titanic_ordinal.head())

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest,pclass_ordinal
0,1,1,"allen, miss. elisabeth walton",female,29.0000,0,0,24160,211.3375,b5,s,2,NaN,"st louis, mo",1
1,1,1,"allison, master. hudson trevor",male,0.9167,1,2,113781,151.5500,c22 c26,s,11,NaN,"montreal, pq / chesterville, on",1
2,1,0,"allison, miss. helen loraine",female,2.0000,1,2,113781,151.5500,c22 c26,s,missing,NaN,"montreal, pq / chesterville, on",1
3,1,0,"allison, mr. hudson joshua creighton",male,30.0000,1,2,113781,151.5500,c22 c26,s,missing,135.0,"montreal, pq / chesterville, on",1
4,1,0,"allison, mrs. hudson j c (bessie waldo daniels)",female,25.0000,1,2,113781,151.5500,c22 c26,s,missing,NaN,"montreal, pq / chesterville, on",1


### Frequency Encoding

Frequency encoding replaces each category with how often it appears.

This can be useful for high-cardinality columns.

In [11]:
# Frequency encoding

titanic_freq = titanic_df.copy()

freq_cols = [
    col for col in categorical_cols
    if titanic_freq[col].nunique() > 2
]

for col in freq_cols:
    freq_map = titanic_freq[col].value_counts(normalize=True)
    titanic_freq[col + "_frequency"] = titanic_freq[col].map(freq_map)

display(titanic_freq[[col for col in titanic_freq.columns if "frequency" in col]].head())

,pclass_frequency,name_frequency,ticket_frequency,cabin_frequency,embarked_frequency,boat_frequency,home.dest_frequency
0,0.246753,0.000764,0.003056,0.001528,0.698243,0.009931,0.003056
1,0.246753,0.000764,0.004584,0.003056,0.698243,0.019099,0.003056
2,0.246753,0.000764,0.004584,0.003056,0.698243,0.628724,0.003056
3,0.246753,0.000764,0.004584,0.003056,0.698243,0.628724,0.003056
4,0.246753,0.000764,0.004584,0.003056,0.698243,0.628724,0.003056


### Target Encoding

Target encoding replaces each category with the average target value for that category.

For Titanic, the target is usually `survived`.

Important: target encoding should be done after train-test split to reduce data leakage.

In [12]:
# Target encoding helper function

def target_encode_train_test(X_train, X_test, y_train, col):
    target_means = y_train.groupby(X_train[col]).mean()
    global_mean = y_train.mean()
    
    X_train[col + "_target_encoded"] = X_train[col].map(target_means)
    X_test[col + "_target_encoded"] = X_test[col].map(target_means).fillna(global_mean)
    
    return X_train, X_test, target_means

In [13]:
# Apply target encoding on Titanic

target_col = "survived"

if target_col in titanic_df.columns:
    X = titanic_df.drop(columns=[target_col])
    y = titanic_df[target_col].astype(int)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    target_encode_cols = ["sex", "embarked"]
    target_encode_cols = [col for col in target_encode_cols if col in X_train.columns]
    
    target_encoding_maps = {}
    
    for col in target_encode_cols:
        X_train, X_test, mapping = target_encode_train_test(X_train, X_test, y_train, col)
        target_encoding_maps[col] = mapping
        print(f"\nTarget encoding map for {col}:")
        display(mapping)
    
    display(X_train[[col for col in X_train.columns if "target_encoded" in col]].head())


Target encoding map for sex:


sex
female    0.720430
male      0.195556
Name: survived, dtype: float64


Target encoding map for embarked:


embarked
c    0.579439
q    0.330097
s    0.331507
Name: survived, dtype: float64

,sex_target_encoded,embarked_target_encoded
999,0.720430,0.330097
392,0.720430,0.579439
628,0.720430,0.331507
1165,0.195556,0.579439
604,0.720430,0.331507


### Final Titanic Encoding Example

Recommended simple approach:

- One-hot encode low-cardinality categorical columns
- Use ordinal encoding for ordered variables like class
- Use target or frequency encoding only when needed

In [14]:
# Final encoded Titanic dataset

titanic_final_encoded = titanic_df.copy()

# One-hot encode low-cardinality categorical columns (exclude target)
final_onehot_cols = [
    col for col in titanic_final_encoded.select_dtypes(
        include=["object", "str", "category", "bool"]
    ).columns
    if titanic_final_encoded[col].nunique() <= 10 and col != "survived"
]

titanic_final_encoded = pd.get_dummies(
    titanic_final_encoded,
    columns=final_onehot_cols,
    drop_first=True,
    dtype=int
)

print("Final encoded Titanic shape:", titanic_final_encoded.shape)
display(titanic_final_encoded.head())

Final encoded Titanic shape: (1309, 17)


,survived,name,age,sibsp,parch,ticket,fare,cabin,boat,body,home.dest,pclass_2,pclass_3,sex_male,embarked_missing,embarked_q,embarked_s
0,1,"allen, miss. elisabeth walton",29.0000,0,0,24160,211.3375,b5,2,NaN,"st louis, mo",0,0,0,0,0,1
1,1,"allison, master. hudson trevor",0.9167,1,2,113781,151.5500,c22 c26,11,NaN,"montreal, pq / chesterville, on",0,0,1,0,0,1
2,0,"allison, miss. helen loraine",2.0000,1,2,113781,151.5500,c22 c26,missing,NaN,"montreal, pq / chesterville, on",0,0,0,0,0,1
3,0,"allison, mr. hudson joshua creighton",30.0000,1,2,113781,151.5500,c22 c26,missing,135.0,"montreal, pq / chesterville, on",0,0,1,0,0,1
4,0,"allison, mrs. hudson j c (bessie waldo daniels)",25.0000,1,2,113781,151.5500,c22 c26,missing,NaN,"montreal, pq / chesterville, on",0,0,0,0,0,1


# Encoding Categorical Variables: LendingClub Dataset

LendingClub data usually contains many categorical variables.

Examples may include:

- grade
- sub_grade
- home_ownership
- verification_status
- purpose
- addr_state
- application_type
- term
- emp_length

Because some columns may have many categories, we will use:

1. One-hot encoding for low-cardinality columns  
2. Ordinal encoding for ordered columns  
3. Frequency encoding for high-cardinality columns  
4. Target encoding for predictive categorical variables  

In [15]:
# Make a working copy

lc_enc = lending_df.copy()

print("Shape:", lc_enc.shape)
display(lc_enc.head())

Shape: (9578, 14)


,credit.policy,purpose,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,0
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,0
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,0
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,0
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,0


In [16]:
# Identify categorical columns

lc_categorical_cols = lc_enc.select_dtypes(
    include=["object", "str", "category", "bool"]
).columns.tolist()

print("Number of categorical columns:", len(lc_categorical_cols))
print(lc_categorical_cols)

cat_summary = pd.DataFrame({
    "column": lc_categorical_cols,
    "unique_values": [lc_enc[col].nunique(dropna=False) for col in lc_categorical_cols],
    "missing_percent": [lc_enc[col].isnull().mean() * 100 for col in lc_categorical_cols]
}).sort_values("unique_values", ascending=False)

display(cat_summary)

Number of categorical columns: 1
['purpose']


,column,unique_values,missing_percent
0,purpose,7,0.0


### Clean Categorical Columns

Before encoding:

- Fill missing values
- Convert to lowercase
- Strip spaces
- Standardize values

In [17]:
# Clean categorical columns

for col in lc_categorical_cols:
    lc_enc[col] = lc_enc[col].astype("object")
    lc_enc[col] = lc_enc[col].fillna("missing")
    lc_enc[col] = lc_enc[col].astype(str).str.strip().str.lower()

display(lc_enc[lc_categorical_cols].head())

,purpose
0,debt_consolidation
1,credit_card
2,debt_consolidation
3,debt_consolidation
4,credit_card


### Group Rare Categories

Rare categories can create too many sparse columns after one-hot encoding.

We group categories below a frequency threshold into `"other"`.

In [18]:
def group_rare_categories(df, col, min_percent=1.0):
    freq = df[col].value_counts(normalize=True) * 100
    common_categories = freq[freq >= min_percent].index
    
    return df[col].where(df[col].isin(common_categories), "other")

In [19]:
# Apply rare category grouping

lc_grouped = lc_enc.copy()

for col in lc_categorical_cols:
    lc_grouped[col] = group_rare_categories(lc_grouped, col, min_percent=1.0)

# Check result for high-cardinality columns
for col in lc_categorical_cols[:10]:
    print("\n", col)
    display(lc_grouped[col].value_counts().head(20))


 purpose


purpose
debt_consolidation    3957
all_other             2331
credit_card           1262
home_improvement       629
small_business         619
major_purchase         437
educational            343
Name: count, dtype: int64

### One-Hot Encoding

Use one-hot encoding for low-cardinality categorical variables.

Good candidates:

- home_ownership
- verification_status
- purpose
- application_type
- term

In [20]:
# Select low-cardinality columns

low_cardinality_cols = [
    col for col in lc_grouped.select_dtypes(
        include=["object", "str", "category", "bool"]
    ).columns
    if lc_grouped[col].nunique() <= 10
]

print("Low-cardinality columns for one-hot encoding:")
print(low_cardinality_cols)

Low-cardinality columns for one-hot encoding:
['purpose']


In [21]:
# One-hot encoding with pandas

lc_onehot = pd.get_dummies(
    lc_grouped,
    columns=low_cardinality_cols,
    drop_first=True,
    dtype=int
)

print("Original shape:", lc_grouped.shape)
print("After one-hot encoding:", lc_onehot.shape)

display(lc_onehot.head())

Original shape: (9578, 14)
After one-hot encoding: (9578, 19)


,credit.policy,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_major_purchase,purpose_small_business
0,1,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,0,0,1,0,0,0,0
1,1,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,0,1,0,0,0,0,0
2,1,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,0,0,1,0,0,0,0
3,1,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,0,0,1,0,0,0,0
4,1,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,0,1,0,0,0,0,0


In [22]:
# One-hot encoding with sklearn

onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    drop="first"
)

encoded_array = onehot_encoder.fit_transform(lc_grouped[low_cardinality_cols])

encoded_col_names = onehot_encoder.get_feature_names_out(low_cardinality_cols)

lc_onehot_sklearn = pd.DataFrame(
    encoded_array,
    columns=encoded_col_names,
    index=lc_grouped.index
)

display(lc_onehot_sklearn.head())

,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_major_purchase,purpose_small_business
0,0.0,1.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0


### Ordinal Encoding

Some LendingClub features are ordered.

Examples:

- grade: a < b < c < d < e < f < g
- sub_grade: a1 < a2 < ... < g5
- emp_length: missing/unknown < <1 year < 1 year < ... < 10+ years

Ordinal encoding preserves this order.

In [23]:
# Ordinal encoding for grade, sub_grade, and emp_length

lc_ordinal = lc_grouped.copy()

ordinal_mappings = {}

if "grade" in lc_ordinal.columns:
    grade_order = ["a", "b", "c", "d", "e", "f", "g", "missing", "other"]
    grade_map = {grade: i for i, grade in enumerate(grade_order)}
    lc_ordinal["grade_ordinal"] = lc_ordinal["grade"].map(grade_map).fillna(-1)
    ordinal_mappings["grade"] = grade_map

if "sub_grade" in lc_ordinal.columns:
    sub_grade_order = [
        f"{grade}{num}"
        for grade in ["a", "b", "c", "d", "e", "f", "g"]
        for num in range(1, 6)
    ]
    sub_grade_order = sub_grade_order + ["missing", "other"]
    sub_grade_map = {sub_grade: i for i, sub_grade in enumerate(sub_grade_order)}
    lc_ordinal["sub_grade_ordinal"] = lc_ordinal["sub_grade"].map(sub_grade_map).fillna(-1)
    ordinal_mappings["sub_grade"] = sub_grade_map

if "emp_length" in lc_ordinal.columns:
    emp_length_order = [
        "missing",
        "< 1 year",
        "1 year",
        "2 years",
        "3 years",
        "4 years",
        "5 years",
        "6 years",
        "7 years",
        "8 years",
        "9 years",
        "10+ years",
        "other"
    ]
    emp_length_map = {value: i for i, value in enumerate(emp_length_order)}
    lc_ordinal["emp_length_ordinal"] = lc_ordinal["emp_length"].map(emp_length_map).fillna(-1)
    ordinal_mappings["emp_length"] = emp_length_map

print("Ordinal mappings:")
for name, mapping in ordinal_mappings.items():
    print(name, mapping)

display(lc_ordinal[[col for col in lc_ordinal.columns if "ordinal" in col]].head())

Ordinal mappings:


""
0
1
2
3
4


### Frequency Encoding

Frequency encoding replaces a category with how often it appears.

This is useful for high-cardinality variables like state, zip code, employer title, or job title.

In [24]:
# Select high-cardinality columns

high_cardinality_cols = [
    col for col in lc_grouped.select_dtypes(
        include=["object", "str", "category", "bool"]
    ).columns
    if lc_grouped[col].nunique() > 10
]

print("High-cardinality columns:")
print(high_cardinality_cols)

High-cardinality columns:
[]


In [25]:
# Frequency encoding

lc_frequency = lc_grouped.copy()

for col in high_cardinality_cols:
    freq_map = lc_frequency[col].value_counts(normalize=True)
    lc_frequency[col + "_frequency"] = lc_frequency[col].map(freq_map)

frequency_cols_created = [col + "_frequency" for col in high_cardinality_cols]

display(lc_frequency[frequency_cols_created].head())

""
0
1
2
3
4


### Target Encoding

Target encoding replaces each category with the average target value for that category.

For LendingClub, a common target is whether the loan defaulted.

Important: target encoding must be done after train-test split to avoid data leakage.

In [26]:
# Smoothed target encoding reduces overfitting on rare categories by pulling
# the category mean toward the global mean proportionally to sample size.
# Formula: (count * category_mean + smoothing * global_mean) / (count + smoothing)

def smooth_target_encode_train_test(X_train, X_test, y_train, col, smoothing=20):
    stats = y_train.groupby(X_train[col]).agg(["mean", "count"])
    global_mean = y_train.mean()

    smoothed = (stats["count"] * stats["mean"] + smoothing * global_mean) / (
        stats["count"] + smoothing
    )

    X_train = X_train.copy()
    X_test = X_test.copy()
    X_train[col + "_target_encoded"] = X_train[col].map(smoothed)
    X_test[col + "_target_encoded"] = X_test[col].map(smoothed).fillna(global_mean)

    return X_train, X_test, smoothed

In [30]:
target_col = "not.fully.paid"

X = lc_enc.drop(columns=[target_col])
y = lc_enc[target_col].astype(int)

print(y.value_counts())
print(y.value_counts(normalize=True))

not.fully.paid
0    8045
1    1533
Name: count, dtype: int64
not.fully.paid
0    0.839946
1    0.160054
Name: proportion, dtype: float64


In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [32]:
target_encode_cols = [
    col for col in ["grade", "sub_grade", "purpose", "addr_state", "home_ownership"]
    if col in X_train.columns
]

for col in target_encode_cols:
    X_train, X_test, mapping = smooth_target_encode_train_test(
        X_train,
        X_test,
        y_train,
        col,
        smoothing=20
    )

    print(f"\nTarget encoding for {col}:")
    display(mapping.sort_values(ascending=False).head(10))


Target encoding for purpose:


purpose
small_business        0.274145
educational           0.185236
home_improvement      0.176517
all_other             0.164854
debt_consolidation    0.153982
credit_card           0.116110
major_purchase        0.102413
dtype: float64

### Final LendingClub Encoding Example

A practical approach:

- Ordinal encode grade, sub_grade, and emp_length
- One-hot encode low-cardinality columns
- Frequency encode high-cardinality columns
- Avoid target encoding unless you are careful about leakage

In [33]:
# Final encoded LendingClub dataset

lc_final_encoded = lc_grouped.copy()

# 1. Ordinal encode ordered columns
if "grade" in lc_final_encoded.columns:
    grade_order = ["a", "b", "c", "d", "e", "f", "g", "missing", "other"]
    grade_map = {grade: i for i, grade in enumerate(grade_order)}
    lc_final_encoded["grade_ordinal"] = lc_final_encoded["grade"].map(grade_map).fillna(-1)

if "sub_grade" in lc_final_encoded.columns:
    sub_grade_order = [
        f"{grade}{num}"
        for grade in ["a", "b", "c", "d", "e", "f", "g"]
        for num in range(1, 6)
    ]
    sub_grade_order = sub_grade_order + ["missing", "other"]
    sub_grade_map = {sub_grade: i for i, sub_grade in enumerate(sub_grade_order)}
    lc_final_encoded["sub_grade_ordinal"] = lc_final_encoded["sub_grade"].map(sub_grade_map).fillna(-1)

if "emp_length" in lc_final_encoded.columns:
    emp_length_order = [
        "missing",
        "< 1 year",
        "1 year",
        "2 years",
        "3 years",
        "4 years",
        "5 years",
        "6 years",
        "7 years",
        "8 years",
        "9 years",
        "10+ years",
        "other"
    ]
    emp_length_map = {value: i for i, value in enumerate(emp_length_order)}
    lc_final_encoded["emp_length_ordinal"] = lc_final_encoded["emp_length"].map(emp_length_map).fillna(-1)

# 2. Frequency encode high-cardinality columns
final_high_card_cols = [
    col for col in lc_final_encoded.select_dtypes(
        include=["object", "str", "category", "bool"]
    ).columns
    if lc_final_encoded[col].nunique() > 10
]

for col in final_high_card_cols:
    freq_map = lc_final_encoded[col].value_counts(normalize=True)
    lc_final_encoded[col + "_frequency"] = lc_final_encoded[col].map(freq_map)

# 3. One-hot encode remaining low-cardinality columns
final_low_card_cols = [
    col for col in lc_final_encoded.select_dtypes(
        include=["object", "str", "category", "bool"]
    ).columns
    if lc_final_encoded[col].nunique() <= 10
]

lc_final_encoded = pd.get_dummies(
    lc_final_encoded,
    columns=final_low_card_cols,
    drop_first=True,
    dtype=int
)

print("Final encoded LendingClub shape:", lc_final_encoded.shape)
display(lc_final_encoded.head())

Final encoded LendingClub shape: (9578, 19)


,credit.policy,int.rate,installment,log.annual.inc,dti,fico,days.with.cr.line,revol.bal,revol.util,inq.last.6mths,delinq.2yrs,pub.rec,not.fully.paid,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_major_purchase,purpose_small_business
0,1,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,0,0,1,0,0,0,0
1,1,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,0,1,0,0,0,0,0
2,1,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,0,0,1,0,0,0,0
3,1,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,0,0,1,0,0,0,0
4,1,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,0,1,0,0,0,0,0
